In [1]:
import os
import sys
from pathlib import Path

# Add the project root containing 'src' to Python's search path
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src").is_dir():
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break

from src.audio.cloud_storage.s3_operations import S3Sync
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.constants import *
from src.audio.entity.artifact_entity import *

In [2]:
class ModelPusher:

    def __init__(self, model_evaluation_artifact: ModelEvaluationArtifacts):
        self.model_evaluation_artifact = model_evaluation_artifact

    def initiate_model_pusher(self):
        try:
            logging.info("Model Pusher Started")
            if self.model_evaluation_artifact.is_model_accepted:
                trained_model_path = self.model_evaluation_artifact.trained_model_path
                s3_model_folder_path = self.model_evaluation_artifact.s3_model_path

                if not s3_model_folder_path.endswith('models/'):
                    s3_model_folder_path = os.path.join(s3_model_folder_path).rstrip('/') + '/models/'

                logging.info(f"Using model path {s3_model_folder_path}")

                s3_sync = S3Sync()

                if not os.path.exists(trained_model_path):
                    raise FileNotFoundError(f"Trained model path {trained_model_path} does not exist.")

                s3_sync.sync_folder_to_s3(
                    folder=trained_model_path,
                    aws_bucket_url=s3_model_folder_path
                )

                message = "Model Pushed to the Production Environment Successfully"
                response = {
                    "is_model_pushed": True,
                    "message": message,
                    "s3_model_path": f"{s3_model_folder_path.rstrip('/')}/{MODEL_NAME}"
                }
                logging.info(response)

            else:
                s3_model_folder_path = self.model_evaluation_artifact.s3_model_path
                if not s3_model_folder_path.endswith('models/'):
                    s3_model_folder_path = os.path.join(s3_model_folder_path).rstrip('/') + '/models/'

                message = "Model is not accepted for pushing to production since there is a better model"
                response = {
                    "is_model_pushed": False,
                    "message": message,
                    "s3_model_path": s3_model_folder_path
                }
                logging.info(response)

            model_pusher_artifact = ModelPusherArtifacts(response=response)
            logging.info("Model Pusher Completed")

            return model_pusher_artifact

        except Exception as e:
            logging.error(f"Error in model pusher: {str(e)}")
            raise CustomException(e, sys) from e